# ESPIRiT on brain — hyperparameter check

Brain currently ships Walsh maps (`brain_T2W_coil_combined`), knee ships ESPIRiT
(`knee_coil_combined/pd`). This runs `physics.smaps.espirit` on a random brain
slice and scores it against the stored Walsh maps on the same data.

The number to tune against is the **null-space residual**
`|| y - S (S^H y) || / || y ||` over the object support: the fraction of the coil
data the maps fail to explain. Maps are only defined up to a per-pixel phase, so
this is the right comparison — an element-wise diff against Walsh is not.

In [ ]:
import json, os, pathlib, sys
import matplotlib.pyplot as plt
import torch

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from datasets.registry import build_loader
from physics.smaps import espirit

# ===================== ESPIRiT hyperparameters — TUNE =====================
ACS_SIZE        = (24, 24)   # calibration region pulled from the k-space centre
KERNEL_SIZE     = 6          # Hankel patch side; 6-8 typical
THRESH_ROWSPACE = 0.05       # keep singular values >= this * S_max
THRESH_EIG      = 0.95       # eigenvalue cut -> defines the support mask
# ==========================================================================

CONFIG = "config/brain/mg/mglpds_R8.json"
SPLIT  = "val"
SEED   = None                # None = a different random slice each run
COILS_SHOWN = 4

cfg = json.load(open(CONFIG))
data_cfg = dict(cfg["data"][SPLIT])
data_cfg["batch_size"] = 1
loader = build_loader(data_cfg, shuffle=True, drop_last=False)
print(f"{CONFIG}  split={SPLIT}  scale_fac={data_cfg.get('scale_fac')}")

In [ ]:
if SEED is not None:
    torch.manual_seed(SEED)

kspace, smaps_walsh, image, _ = next(iter(loader))
kspace = kspace.to(torch.complex64)
smaps_walsh = smaps_walsh.to(torch.complex64)
if kspace.dim() == 3:                      # (C, H, W) -> (1, C, H, W)
    kspace, smaps_walsh = kspace[None], smaps_walsh[None]
B, C, H, W = kspace.shape
print(f"kspace {tuple(kspace.shape)}  walsh {tuple(smaps_walsh.shape)}  "
      f"image {tuple(image.shape)}")

smaps_e = espirit(kspace, acs_size=ACS_SIZE, kernel_size=KERNEL_SIZE,
                  thresh_rowspace=THRESH_ROWSPACE, thresh_eig=THRESH_EIG)

# fastMRI k-space is centre-DC (operators/fourier.py::ifftc is the centred
# convention), which is what espirit's centre-slice ACS extraction assumes.
coil_imgs = torch.fft.fftshift(
    torch.fft.ifft2(torch.fft.ifftshift(kspace, dim=(-2, -1)),
                    norm="ortho"), dim=(-2, -1))


def score(S, tag):
    '''Null-space residual + unit-norm + support, over the object only.'''
    n = S.abs().pow(2).sum(1, keepdim=True).sqrt()          # (B,1,H,W)
    sup = n[:, 0] > 1e-6
    frac = float(sup.float().mean())
    # project the coil images onto span(S) at each pixel
    proj = S * (S.conj() * coil_imgs).sum(1, keepdim=True) / (n.pow(2) + 1e-12)
    num = (coil_imgs - proj).abs().pow(2).sum(1)[sup].sum()
    den = coil_imgs.abs().pow(2).sum(1)[sup].sum()
    res = float((num / (den + 1e-12)).sqrt())
    inside = n[:, 0][sup]
    print(f"  {tag:<10} support={frac:6.1%}  ||s||: mean={float(inside.mean()):.4f} "
          f"min={float(inside.min()):.4f} max={float(inside.max()):.4f}  "
          f"null-space residual={res:.4f}")
    return sup, res


print(f"\nACS={ACS_SIZE} kernel={KERNEL_SIZE} rowspace={THRESH_ROWSPACE} eig={THRESH_EIG}")
sup_e, res_e = score(smaps_e, "espirit")
sup_w, res_w = score(smaps_walsh, "walsh")
print(f"\n  espirit explains {'MORE' if res_e < res_w else 'LESS'} of the coil data "
      f"than the stored walsh maps ({res_e:.4f} vs {res_w:.4f})")

comb_e = (smaps_e.conj() * coil_imgs).sum(1, keepdim=True)
comb_w = (smaps_walsh.conj() * coil_imgs).sum(1, keepdim=True)
rss = coil_imgs.abs().pow(2).sum(1, keepdim=True).sqrt()

In [ ]:
# Fixed windows throughout -- no per-image percentiles. Magnitude maps are
# unit-norm by construction so [0, 1] is the natural window; the image row
# shares ONE window taken from the RSS so the three are comparable.
VIMG = float(rss.abs().max())
b = 0
fig, ax = plt.subplots(3, COILS_SHOWN, figsize=(3.0 * COILS_SHOWN, 8.6))
picks = torch.linspace(0, C - 1, COILS_SHOWN).long().tolist()

for j, c in enumerate(picks):
    ax[0][j].imshow(smaps_e[b, c].abs().cpu(), cmap="gray", vmin=0, vmax=1)
    ax[0][j].set_title(f"coil {c}", fontsize=9)
    ax[1][j].imshow(smaps_walsh[b, c].abs().cpu(), cmap="gray", vmin=0, vmax=1)
ax[0][0].set_ylabel("|s| espirit", fontsize=10)
ax[1][0].set_ylabel("|s| walsh", fontsize=10)

panels = [(sup_e[b].float().cpu(), "espirit support", dict(cmap="gray", vmin=0, vmax=1)),
          (comb_e[b, 0].abs().cpu(), "espirit combined", dict(cmap="gray", vmin=0, vmax=VIMG)),
          (comb_w[b, 0].abs().cpu(), "walsh combined", dict(cmap="gray", vmin=0, vmax=VIMG)),
          (rss[b, 0].abs().cpu(), "RSS", dict(cmap="gray", vmin=0, vmax=VIMG))]
for j in range(COILS_SHOWN):
    if j < len(panels):
        im, t, kw = panels[j]
        ax[2][j].imshow(im, **kw); ax[2][j].set_title(t, fontsize=9)
    else:
        ax[2][j].axis("off")
for row in ax:
    for a in row:
        a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"ESPIRiT  ACS={ACS_SIZE} k={KERNEL_SIZE} "
             f"rowspace={THRESH_ROWSPACE} eig={THRESH_EIG}   "
             f"residual {res_e:.4f} (walsh {res_w:.4f})", y=1.01)
fig.tight_layout(); plt.show()

## Sweep

`THRESH_EIG` sets the support and `ACS_SIZE` sets how much calibration data the
kernel sees — those two move the residual most.

In [ ]:
print(f"{'acs':>9}{'kernel':>8}{'eig':>7}{'support':>10}{'residual':>11}")
for acs in [(16, 16), (24, 24), (32, 32)]:
    for keig in [0.90, 0.95, 0.98]:
        S = espirit(kspace, acs_size=acs, kernel_size=KERNEL_SIZE,
                    thresh_rowspace=THRESH_ROWSPACE, thresh_eig=keig)
        n = S.abs().pow(2).sum(1, keepdim=True).sqrt()
        sup = n[:, 0] > 1e-6
        proj = S * (S.conj() * coil_imgs).sum(1, keepdim=True) / (n.pow(2) + 1e-12)
        num = (coil_imgs - proj).abs().pow(2).sum(1)[sup].sum()
        den = coil_imgs.abs().pow(2).sum(1)[sup].sum()
        r = float((num / (den + 1e-12)).sqrt())
        print(f"{str(acs):>9}{KERNEL_SIZE:>8}{keig:>7.2f}"
              f"{float(sup.float().mean()):>9.1%}{r:>11.4f}")
print(f"\nwalsh reference: residual={res_w:.4f}, support={float(sup_w.float().mean()):.1%}")